# CIS I — Assignment 1: Pivot Calibration

This notebook walks through the pivot-calibration scenario from the
assignment (Figure 1): a pointer and a calibration object, each
carrying a marker body, observed by an optical tracker. Your job is to
**design** the marker-sphere layouts and workspace placements, then
**calibrate** the pointer's tip position `p_tip` to within about 2mm.

Everything runs against `src.simulation`, a self-contained simulator that plays the role of the physical tracker +
hand-guided robot + calibration sensor described in the handout. It injects measurement noise — your code never sees ground truth
except where explicitly noted for **validation/reporting** in Step 8 (real
hardware wouldn't give you that; the point there is to report your achieved
accuracy, not to use the ground truth in your calibration math itself).



In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
from src import simulation
from src import vct3, Rot, Frame

np.set_printoptions(precision=3, suppress=True)


In [2]:
#constants
N_TEST_POSES = 100
TIP_ERROR_SPEC_MM = 2
np.random.seed(0)  # for reproducibility

In [3]:
from src.simulation import constants
print({k: getattr(constants, k) for k in dir(constants) if k.isupper()})

{'BODY_PLACEMENT_ANGLE_STD': 0.03577817531305904, 'BODY_PLACEMENT_POS_STD': 8.94454382826476, 'CAL_SENSOR_NOISE_STD': 0.002, 'CAL_SENSOR_RANGE_MM': 5.0, 'HAND_GUIDE_ANGLE_STD': 0.05, 'HAND_GUIDE_POS_STD': 1.7889087656529516, 'MARKER_MANUFACTURING_STD': 0.7155635062611807, 'SAMPLE_DELTA_T_SEC': 0.03, 'SENSOR_NOISE_STD': 0.02, 'TEST_POSE_ANGLE_STD': 0.02, 'TEST_POSE_POS_STD': 0.7155635062611807, 'TRACKER_JIGGLE_ANGLE_STD': 0.01, 'TRACKER_JIGGLE_POS_STD': 5.0, '_CHI3_95': 2.795}


## Step 1: Design the pointer's marker body

Choose approximate local marker-sphere positions `a_ptr,k` (a `List[vct3]`,
relative to the pointer's own coordinate frame) and the pointer tip's
approximate local position `p_tip_approx`. These are your
*nominal design values* — the simulator will independently perturb the
*actual* manufactured positions by a small, noise amount (bounded at
2mm).


In [4]:
ptr_local_positions = [
    # TODO: at least 4 non-coplanar vct3 points, e.g. vct3(x, y, z)
    vct3(0, 0, 0),
    vct3(80, 0, 0),
    vct3(0, 60, 0),
    vct3(20, 30, 50)
]
ptr_tip_approx = vct3(0, 0, -50) #50mm below the marker cluster  # TODO: vct3, the tip's approximate position in the pointer's local frame


## Step 1.1: Design the calibration object's marker body

Same idea, for the calibration object's local marker positions `a_cal,k`.


In [5]:
cal_local_positions = [
    # TODO: at least 4 non-coplanar vct3 points
    vct3(0, 0, 0),
    vct3(100, 0, 0),
    vct3(0, 90, 0),
    vct3(30, 40, 60)
]


## Step 2: Approximate workspace poses

Specify approximate poses (as `Frame` class from src) for the tracking camera
(`F_tracker`) and the calibration object (`F_cal`), relative to the
workspace. The tracker's usable volume is the tetrahedral region in the
handout's Figure 2 (roughly 100-200cm in front of the tracker, widening
from ~120cm to ~200cm across) — place the calibration object, and later
the pointer, so they're inside that volume.


In [6]:
F_tracker_approx = Frame(Rot(), vct3(300, 300, -1500))  # TODO: Frame, tracker pose relative to the workspace
F_cal_approx = Frame(Rot(), vct3(300, 300, 300))  # TODO: Frame, calibration object pose relative to the workspace


## Step 3: Build the simulation

Fully provided — this just wires up your Step 1/2 design choices through
`simulation`'s `DefineMarkerBody` / `PlaceMarkerBody` / `CreateTracker`
equivalents. `ptr_placed`/`cal_placed` are `PlacedBody` handles: each pairs
a `MarkerBody` with its hidden actual placement, and gets passed to
`simulation.sample_marker_bodies(...)` whenever you want a reading.


In [7]:
ptr_mb = simulation.define_marker_body("ptr_mb", ptr_local_positions)
cal_mb = simulation.define_marker_body("cal_mb", cal_local_positions)

cal_placed = simulation.place_marker_body(cal_mb, F_cal_approx)
# The pointer's own body doesn't have a meaningful "workspace placement" yet
# It moves every time it's hand-guided (Step 4-6). 
# precision="precise" matches a robot-controlled pose rather than a coarsely, manually placed
# static object (see place_marker_body's docstring).
ptr_placed = simulation.place_marker_body(ptr_mb, Frame.eye(), precision="precise")

ptr = simulation.define_pointer("ptr1", ptr_tip_approx, ptr_mb)
trk = simulation.create_tracker("trk1", F_tracker_approx)

print("simulation built:", ptr_mb.name, cal_mb.name, ptr.name, trk.name)


simulation built: ptr_mb cal_mb ptr1 trk1


## Step 3.1: Refine `a_ptr,k` / `a_cal,k`

> "Observations of marker positions can be used to determine accurate
> estimates of relative positions a_ptr,k of markers on pointer and
> calibration objects... **Hint**: You probably want to take multiple
> readings in order to reduce the uncertainty associated with the a_k
> values."


In [8]:
def refine_marker_body(trk, placed, n_readings=30, passes=3):
    """
    TODO: implement.

    input:
        trk - Tracker to read with
        placed - PlacedBody whose body's nominal marker positions should be
            refined in place (via body.set_marker_positions(...))
        n_readings - int, readings to average per pass
        passes - int, how many times to repeat
    output: None (mutates placed.body's nominal_marker_positions in place)
    """
    body = placed.body
    for i in range(passes):
        readings = []
        for j in range(n_readings):
            obs = simulation.sample_marker_bodies(trk, [placed])[body.name]         #one pose for the body, and position of the 4 markers in the tracker frame
            F_inv = obs.frame.inv()             #converts points from the tracker frame to the body frame
            #print(F_inv)

            this_reading = []
            for m in obs.markers:           #obs.markers = list of the markers
                b = m.p             #b = the marker position in the tracker frame
                a = F_inv * b       #a = the same marker positions in the body frame
                this_reading.append(np.array(a.vec).reshape(3))
            readings.append(this_reading)
        
        averages = np.mean(readings, axis = 0)
        #print(averages)
        body.set_marker_positions([vct3(*row) for row in averages])
        print(f"{body.name} pass {i + 1} done")



    #raise NotImplementedError("TODO: implement marker-body refinement (see markdown above)")
    



refine_marker_body(trk, ptr_placed)
refine_marker_body(trk, cal_placed)

# print("ptr_mb refined:\n", np.array([np.array(a.vec).reshape(3) for a in ptr_mb.nominal_marker_positions]))
# print("cal_mb refined:\n", np.array([np.array(a.vec).reshape(3) for a in cal_mb.nominal_marker_positions]))

ptr_mb pass 1 done
ptr_mb pass 2 done
ptr_mb pass 3 done
cal_mb pass 1 done
cal_mb pass 2 done
cal_mb pass 3 done


## Steps 4-6: Collect pivot-calibration observations

For each observation `j`: hand-guide the pointer near the calibration
object at some desired orientation `R_desired,j`, take one simultaneous
tracker reading of *both* bodies, and read the calibration sensor. Loop
skeleton is provided; you choose how many observations to take and how to
vary `R_desired,j`


In [9]:
N_OBSERVATIONS = 50  # TODO

F_ptr_list = []
F_cal_list = []
sensed_tip_list = []
for j in range(N_OBSERVATIONS):
    R_desired = Rot.rand_xyz(np.pi/4, np.pi/4, np.pi)  # TODO: vary the desired pointer orientation across observations
    hg = simulation.hand_guide_pointer(ptr, ptr_placed, cal_placed, R_desired)
    obs = simulation.sample_marker_bodies(trk, [ptr_placed, cal_placed])
    sensed = simulation.sense_tip(ptr, hg)

    F_ptr_list.append(obs["ptr_mb"].frame)
    F_cal_list.append(obs["cal_mb"].frame)
    sensed_tip_list.append(sensed)

print(f"collected {len(F_ptr_list)} observations")


collected 50 observations


`sensed_tip_list` holds each observation's high-accuracy calibration-sensor
reading (or `None` if that observation ended up outside the sensor's 5mm
range) — useful as an independent sanity check on your calibration later,
since it measures the tip position a completely different way (directly,
rather than through the pointer's marker geometry).


## Step 7: Compute a calibrated value for `p_tip`

This is the assignment's core deliverable — implement it yourself and specify the details of implementation in the report.

**Corner case to consider**: What are the corner cases you want to analyse here?

**Minimum data**: how many observations, and what about their rotations,
are needed for the stacked system to actually be solvable? How do you define whether a system is solvable here? Include that in your report


In [10]:
def compute_pivot_calibration(F_ptr_list, F_cal_list):
    """
    TODO: implement.

    input:
        F_ptr_list - List[Frame], the pointer marker body's pose relative to
            the tracker for each observation j (Fptr,j)
        F_cal_list - List[Frame], the calibration object's pose relative to
            the tracker for each observation j (Fcal,j), same order/length
    output:
        p_tip - vct3, the calibrated tip position in the pointer's local frame
        cov_estimate - np.ndarray (3x3), an estimated covariance for p_tip
            (e.g. from the least-squares fit's residuals)
    """
    A_blocks = []
    B_blocks = []

    for i in range(N_OBSERVATIONS):
        F_ptr = F_ptr_list[i]
        F_cal = F_cal_list[i]

        F_j = F_cal.inv() * F_ptr # pointer pose in the cal frame
        R_j = F_j.R.matrix          # 3x3 rotation
        p_j = F_j.p.vec.ravel()     # 3x1 column

        A_blocks.append(np.hstack((R_j, -np.eye(3))))       # [R-j  -I] block, 3x6
        B_blocks.append(-p_j)                               # one -p_j block, length 3

    A = np.vstack(A_blocks)  # 3N x 6
    B = np.concatenate(B_blocks)  # 3N x 1

    solution = np.linalg.inv(A.T @ A) @ A.T @ B  # [p_tip; p_pivot]
    p_tip_vec = solution[:3]

    residual = A @ solution - B
    num_unknowns = 6
    residual_variance = (residual @ residual) / (len(B) - num_unknowns)
    cov = residual_variance * np.linalg.inv(A.T @ A)
    cov_estimate = cov[:3, :3]

    return vct3(*p_tip_vec), cov_estimate

                                       
        
    # raise NotImplementedError("TODO: implement pivot calibration (see markdown above)")

p_tip_est, p_tip_cov = compute_pivot_calibration(F_ptr_list, F_cal_list)


In [11]:
print("calibrated p_tip:", p_tip_est)
print("estimated covariance (mm^2):\n", p_tip_cov)
print("estimated std-dev per axis (mm):", np.sqrt(np.diag(p_tip_cov)))


calibrated p_tip: vct3(x=-0.87, y=-0.16, z=-49.34)
Internal Vector:
[[ -0.873]
 [ -0.164]
 [-49.339]]
estimated covariance (mm^2):
 [[ 0.069 -0.001 -0.003]
 [-0.001  0.071  0.017]
 [-0.003  0.017  0.221]]
estimated std-dev per axis (mm): [0.263 0.267 0.47 ]


In [12]:
sensor_errors = []

for  i in range(len(F_ptr_list)):
    sensed = sensed_tip_list[i]
    if sensed is None:
        continue

    F_j = F_cal_list[i].inv() * F_ptr_list[i]
    predicted = F_j * p_tip_est
    difference = predicted.vec.ravel() - sensed.vec.ravel()
    sensor_errors.append(np.linalg.norm(difference))


print("observations checked: ", len(sensor_errors))
print("mean distace to sensor reading (mm): ", np.mean(sensor_errors))
print("max distance to sensor reading (mm): ", np.max(sensor_errors))
print("min distance to sensor reading (mm): ", np.min(sensor_errors))

observations checked:  49
mean distace to sensor reading (mm):  1.1740947886770288
max distance to sensor reading (mm):  1.2909608879031644
min distance to sensor reading (mm):  0.9264877146747645


## Step 8: Validate against test poses

> "**Hint**: You may want to consider placing additional marker bodies
> close to (but not inside) the cube defined by the test volume. However,
> this will also depend on your design for and placement of the
> calibration object."

The nominal test-pose grid is `theta_x, theta_y, theta_z` each in
`{0, +-pi/6, +-pi/3, +-pi/2}` and `p_x, p_y, p_z` each in
`{300, 300+-100, 300+-200}` mm. For each test pose, the assignment requires a
**single observation only** (no averaging).

Think about why the hint suggests an *additional*, nearby reference body?

In [13]:
near_local_positions = [
    # TODO: at least 4 non-coplanar vct3 points for this reference body
    vct3(-75, -75, 0),
    vct3(75, -75, 0),
    vct3(0, 75, 0),
    vct3(0, 0, 100)
]
F_near_approx = Frame(Rot(), vct3(300, 300, 550))   # TODO: Frame, placed close to (not inside) the test-pose cube

near_mb = simulation.define_marker_body("near_mb", near_local_positions)
near_placed = simulation.place_marker_body(near_mb, F_near_approx)
refine_marker_body(trk, near_placed)


near_mb pass 1 done
near_mb pass 2 done
near_mb pass 3 done


Refine this new body's geometry the same way you refined the pointer's and
calibration object's (Step 3b) before using it.


Now the single-observation test loop. For each nominal test pose `F_t`:
place the pointer there (small, robot-precision placement error, same as
the real system), take one simultaneous reading of the pointer and your
Step-8 reference body, and predict the tip's position (relative to that
reference body) using your calibrated `p_tip`.

**Ground truth for reporting only**: `PlacedBody.actual_placement` and
`MarkerBody.actual_marker_positions` hold the hidden values the simulator
sampled — you have access to them here only because this notebook doesn't
gate them behind a separate eval mode (per the handout: "During debugging
you will have access to all of the ... values. These will not be available
during evaluation."). Use them *only* to compute and report your achieved
error in this section — using them anywhere in your actual calibration math
(Steps 3b/7) kills the point of the assignment.


In [ ]:
thetas = [0, np.pi / 6, -np.pi / 6, np.pi / 3, -np.pi / 3, np.pi / 2, -np.pi / 2]
offsets = [0, 100, -100, 200, -200]
# TODO: the full grid above is large (7^3 * 5^3 = 42,875 combinations) --
# choose a representative subset of (position, orientation) test poses
# rather than every combination.

test_poses = [
    # TODO: build a list of Frame(Rot.xyz(rx, ry, rz), vct3(300+tx, 300+ty, 300+tz))
    # from thetas/offsets above
    # Frame(Rot.xyz(0, 0, 0), vct3(300, 300, 300)),
    # Frame(Rot.xyz(0, , 0), vct3(300, 300, 300)),
]

rng = np.random.default_rng(0)

chosen = set()
while len(test_poses) < N_TEST_POSES:
    pick = (rng.choice(thetas), rng.choice(thetas), rng.choice(thetas),
            rng.choice(offsets), rng.choice(offsets), rng.choice(offsets))
    if pick in chosen:
        continue 
    chosen.add(pick)
    rx, ry, rz, tx, ty, tz = pick
    test_poses.append(Frame(Rot.xyz(rx, ry, rz), vct3(300 + tx, 300 + ty, 300 + tz)))

tip = ptr.get_actual_tip_position()
true_tip_local = vct3(tip.x, tip.y, tip.z)

test_results = []  # list of (F_t_nominal, error_mm)
for F_t_nominal in test_poses:
    # TODO, for each F_t_nominal:
    #   1. test_placed = simulation.place_marker_body(ptr_mb, F_t_nominal, precision="precise")
    test_placed = simulation.place_marker_body(ptr_mb, F_t_nominal, precision="precise")

    #   2. obs = simulation.sample_marker_bodies(trk, [test_placed, near_placed])  -- single reading
    obs = simulation.sample_marker_bodies(trk, [test_placed, near_placed])
    F_ptr = obs["ptr_mb"].frame
    F_near = obs["near_mb"].frame
    
    #   3. predict the tip position relative to near_placed using p_tip_est
    tip_predicted = F_near.inv() * (F_ptr * p_tip_est)

    #   4. compute the TRUE tip position relative to near_placed using
    F_real_true = near_placed.actual_placement.inv() * test_placed.actual_placement
    tip_true = F_real_true * true_tip_local

    #      test_placed.actual_placement / near_placed.actual_placement (ground truth,
    #      for reporting only -- see markdown above)
    #   5. record the error and append (F_t_nominal, error) to test_results
    error = np.linalg.norm(tip_predicted.vec.ravel() - tip_true.vec.ravel())
    test_results.append((F_t_nominal, error))

   
errors = []                                        
for i in range(len(test_results)):
    errors.append(test_results[i][1])
errors = np.array(errors)

print("test poses:  ", len(errors))
print("mean error (mm):  ", round(np.mean(errors)))
print("max error(mm):  ", round(np.max(errors)))

test poses:   100
mean error (mm):   2
max error(mm):   3


In [15]:
z_levels = [100, 200, 300, 400, 500]

print("z (mm) | poses | mean error (mm) | max error (mm)")
for z in z_levels:
    errors_at_z = []
    for i in range(len(test_results)):
        F_t_nominal = test_results[i][0]
        error = test_results[i][1]
        if F_t_nominal.p.vec.ravel()[2] == z:
            errors_at_z.append(error)

    if len(errors_at_z) == 0:
        print(z, "| 0 poses")
        continue
    errors_at_z = np.array(errors_at_z)
    print(z, "|", len(errors_at_z), "|", round(np.mean(errors_at_z), 3), "|", round(np.max(errors_at_z), 3))

z (mm) | poses | mean error (mm) | max error (mm)
100 | 23 | 2.645 | 3.437
200 | 12 | 2.162 | 2.734
300 | 27 | 1.566 | 2.412
400 | 14 | 1.344 | 2.273
500 | 24 | 1.188 | 2.299


## Achieved performance
Summarize the tip-position error across the test-pose grid against the
handout's 2mm spec.


In [ ]:
# TODO: summarize test_results -- e.g. mean/max error, and how many poses
# meet the assignment's ||delta p_t|| <= 2mm spec.

def summarize_tip_errors(error_norms, spec_mm=TIP_ERROR_SPEC_MM):
    error_norms = np.asarray(error_norms, dtype=float)
    return{
        'mean_mm': error_norms.mean(),
        'rms_mm': np.sqrt(np.mean(error_norms**2)),
        'max_mm': error_norms.max(),
        'std_mm': error_norms.std(),
        'worst_pose_index': int(error_norms.argmax()),
        'num_passing': int(np.sum(error_norms <= spec_mm)),
        'num_poses': len(error_norms),
    }


error_norms = [r[1] for r in test_results]
summary = summarize_tip_errors(error_norms)

print(f"Poses meeting {TIP_ERROR_SPEC_MM} mm spec: "
      f"{summary['num_passing']} /{summary['num_poses']}")
print(f"Mean error: {summary['mean_mm']: .3f} mm")
print(f"RMS error: {summary['rms_mm']: .3f} mm")
print(f"Max error: {summary['max_mm']: .3f} mm (pose {summary['worst_pose_index']})")
print(f"Std dev {summary['std_mm']: .3f} mm")

Poses meeting 2 mm spec: 63 /100
Mean error:  1.764 mm
RMS error:  1.889 mm
Max error:  3.437 mm (pose 23)
Std dev  0.676 mm
